<a href="https://colab.research.google.com/github/Srigayathri26/AI-Smart-Rental-Accommodation-Finder/blob/main/AI_Smart_Rental_Accommodation_Finder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai langgraph chromadb networkx pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [ ]:
try:
    for m in client.models.list():
        print(m.id)
except Exception as e:
    print("Could not list models:", e)

Could not list models: name 'client' is not defined


In [ ]:
!pip install -q openai langgraph chromadb networkx pandas

In [ ]:
try:
    for m in client.models.list():
        print(m.id)
except Exception as e:
    print("Could not list models:", e)

Could not list models: name 'client' is not defined


In [ ]:
import requests
from google.colab import userdata

KEY = userdata.get("navi_key")
BASE = userdata.get("base_url").rstrip("/")

for path in ["/models", "/v1/models"]:
    try:
        r = requests.get(BASE + path, headers={"Authorization": f"Bearer {KEY}"}, timeout=30)
        print(path, "->", r.status_code)
        print(r.text[:1500])
        print("-" * 40)
    except Exception as e:
        print(path, "-> error:", str(e)[:150])

/models -> 200
{"data":[{"id":"nano-banana","object":"model","created":1677610602,"owned_by":"openai","mode":"image_generation","max_input_tokens":65536,"max_output_tokens":32768},{"id":"whisper-1","object":"model","created":1677610602,"owned_by":"openai","mode":"audio_transcription"},{"id":"nova-micro","object":"model","created":1677610602,"owned_by":"openai","mode":"chat","max_input_tokens":128000,"max_output_tokens":10000},{"id":"text-embedding-3-small","object":"model","created":1677610602,"owned_by":"openai","mode":"embedding","max_input_tokens":8191},{"id":"gpt-4o-mini-tts","object":"model","created":1677610602,"owned_by":"openai","mode":"audio_speech"},{"id":"gpt-4.1-nano","object":"model","created":1677610602,"owned_by":"openai","mode":"chat","max_input_tokens":1047576,"max_output_tokens":32768}],"object":"list"}
----------------------------------------
/v1/models -> 200
{"data":[{"id":"nano-banana","object":"model","created":1677610602,"owned_by":"openai","mode":"image_generat

In [ ]:
from google.colab import userdata
from openai import OpenAI
import time

BASE = userdata.get("base_url").rstrip("/")
client = OpenAI(api_key=userdata.get("navi_key"), base_url=BASE)

MODELS = ["gpt-4.1-nano", "nova-micro"]   # first one is used; second is the backup

def ask_llm(prompt, retries=3):
    last_error = None
    for model in MODELS:
        for attempt in range(retries):
            try:
                r = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                )
                return r.choices[0].message.content
            except Exception as e:
                last_error = e
                msg = str(e)
                if any(c in msg for c in ("429", "500", "502", "503")) and attempt < retries - 1:
                    time.sleep(2 ** attempt * 3)    # busy server: wait and retry
                    continue
                break                               # other error: try the next model
    raise last_error

print("Client ready")

Client ready


In [ ]:
print(ask_llm("Say hello in one short line"))

Hello!


In [ ]:
import pandas as pd

listings = pd.DataFrame([
 dict(id="L1", name="Sri Sai PG", area="Guindy", rent=5500, wifi=True,  food=True,  lat=13.0067, lon=80.2206, available=True,  phone="98400-00001", desc="Girls PG near Guindy with Wi-Fi, 3 meals, laundry"),
 dict(id="L2", name="Green Nest Hostel", area="Guindy", rent=4800, wifi=True,  food=False, lat=13.0100, lon=80.2150, available=True,  phone="98400-00002", desc="Budget hostel with Wi-Fi, no food, study hall"),
 dict(id="L3", name="Comfort PG", area="Velachery", rent=6500, wifi=True,  food=True,  lat=12.9815, lon=80.2180, available=True,  phone="98400-00003", desc="AC rooms, Wi-Fi, homemade food"),
 dict(id="L4", name="Student Stay", area="Saidapet", rent=5200, wifi=True,  food=True,  lat=13.0213, lon=80.2231, available=False, phone="98400-00004", desc="Shared rooms, Wi-Fi, food included, near metro"),
 dict(id="L5", name="Anna Nagar Rooms", area="Anna Nagar", rent=5900, wifi=False, food=True,  lat=13.0850, lon=80.2101, available=True,  phone="98400-00005", desc="Food provided, no Wi-Fi, quiet area"),
 dict(id="L6", name="Campus View PG", area="Guindy", rent=5800, wifi=True,  food=True,  lat=13.0050, lon=80.2250, available=True,  phone="98400-00006", desc="Walking distance to college, Wi-Fi, food, CCTV"),
])

colleges = {"Anna University": (13.0108, 80.2354)}
listings

,id,name,area,rent,wifi,food,lat,lon,available,phone,desc
0,L1,Sri Sai PG,Guindy,5500,True,True,13.0067,80.2206,True,98400-00001,"Girls PG near Guindy with Wi-Fi, 3 meals, laundry"
1,L2,Green Nest Hostel,Guindy,4800,True,False,13.0100,80.2150,True,98400-00002,"Budget hostel with Wi-Fi, no food, study hall"
2,L3,Comfort PG,Velachery,6500,True,True,12.9815,80.2180,True,98400-00003,"AC rooms, Wi-Fi, homemade food"
3,L4,Student Stay,Saidapet,5200,True,True,13.0213,80.2231,False,98400-00004,"Shared rooms, Wi-Fi, food included, near metro"
4,L5,Anna Nagar Rooms,Anna Nagar,5900,False,True,13.0850,80.2101,True,98400-00005,"Food provided, no Wi-Fi, quiet area"
5,L6,Campus View PG,Guindy,5800,True,True,13.0050,80.2250,True,98400-00006,"Walking distance to college, Wi-Fi, food, CCTV"


In [ ]:
import chromadb

chroma = chromadb.Client()
col = chroma.get_or_create_collection("listings")
col.add(
    ids=listings.id.tolist(),
    documents=(listings.name + ". " + listings.desc).tolist(),
    metadatas=[{"rent": int(r)} for r in listings.rent],
)
print("Stored:", col.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 26.0MiB/s]


Stored: 6


In [ ]:
import networkx as nx

G = nx.Graph()
G.add_edge("Anna University", "Guindy", relation="near")
G.add_edge("Anna University", "Saidapet", relation="near")
G.add_edge("Anna University", "Velachery", relation="connected_by_bus")
for _, r in listings.iterrows():
    G.add_edge(r.area, r.id, relation="has_listing")

def graph_candidates(college):
    ids = set()
    for area in G.neighbors(college):
        ids |= {n for n in G.neighbors(area) if n.startswith("L")}
    return ids

print(graph_candidates("Anna University"))

{'L3', 'L6', 'L1', 'L4', 'L2'}


In [ ]:
import json, math, re
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

class State(TypedDict, total=False):
    query: str
    req: dict
    results: List[dict]
    answer: str

def requirement_agent(s):
    p = f"""Extract from this request and reply with JSON only, no other text.
Keys: college (string), max_km (number), max_rent (number), wifi (true/false), food (true/false).
Request: {s['query']}"""
    txt = ask_llm(p)
    req = json.loads(re.search(r"\{.*\}", txt, re.S).group())
    for name in colleges:
        if name.lower() in str(req.get("college", "")).lower():
            req["college"] = name
    return {"req": req}

def search_agent(s):
    r = s["req"]
    res = col.query(query_texts=[s["query"]], n_results=6,
                    where={"rent": {"$lte": r["max_rent"]}})
    ids = res["ids"][0]
    cands = graph_candidates(r["college"])          # Graph RAG step
    ids = [i for i in ids if i in cands]
    return {"results": listings[listings.id.isin(ids)].to_dict("records")}

def haversine(a, b):
    R = 6371
    la1, lo1, la2, lo2 = map(math.radians, [*a, *b])
    h = math.sin((la2-la1)/2)**2 + math.cos(la1)*math.cos(la2)*math.sin((lo2-lo1)/2)**2
    return 2*R*math.asin(math.sqrt(h))

def location_agent(s):
    c = colleges[s["req"]["college"]]
    out = []
    for x in s["results"]:
        x["km"] = round(haversine(c, (x["lat"], x["lon"])), 2)
        if x["km"] <= s["req"]["max_km"]:
            out.append(x)
    return {"results": out}

def price_agent(s):
    r = s["req"]
    out = [x for x in s["results"]
           if (not r.get("wifi") or x["wifi"]) and (not r.get("food") or x["food"])]
    return {"results": out}

def verification_agent(s):
    return {"results": [x for x in s["results"] if x["available"] and x["phone"]]}

def recommendation_agent(s):
    ranked = sorted(s["results"], key=lambda x: (x["rent"], x["km"]))
    if not ranked:
        return {"answer": "No matching accommodation found. Try a higher budget or distance."}
    slim = [{k: x[k] for k in ("name","area","rent","km","wifi","food","phone")} for x in ranked]
    p = f"Student asked: {s['query']}\nRank these options best to worst with one short reason each, and include rent, distance and phone:\n{slim}"
    return {"answer": ask_llm(p)}

g = StateGraph(State)
for name, fn in [("requirement", requirement_agent), ("search", search_agent),
                 ("location", location_agent), ("price", price_agent),
                 ("verify", verification_agent), ("recommend", recommendation_agent)]:
    g.add_node(name, fn)
g.set_entry_point("requirement")
g.add_edge("requirement", "search"); g.add_edge("search", "location")
g.add_edge("location", "price");     g.add_edge("price", "verify")
g.add_edge("verify", "recommend");   g.add_edge("recommend", END)
app = g.compile()
print("Workflow ready")

Workflow ready


In [ ]:
q = "Find a PG near Anna University within 3 km, below 6000 rupees, with Wi-Fi and food."
out = app.invoke({"query": q})
print(out["req"])
print(out["answer"])

{'college': 'Anna University', 'max_km': 3, 'max_rent': 6000, 'wifi': True, 'food': True}
Sure, here is the ranking of the provided options from best to worst based on the given criteria:

1. **Sri Sai PG**
   - **Rent**: 5500 rupees (below 6000)
   - **Distance**: 1.67 km (within 3 km)
   - **Wi-Fi**: Available
   - **Food**: Available
   - **Phone**: 98400-00001
   - **Reason**: Slightly lower rent, and the price is the most affordable within the specified criteria.

2. **Campus View PG**
   - **Rent**: 5800 rupees (below 6000)
   - **Distance**: 1.3 km (within 3 km)
   - **Wi-Fi**: Available
   - **Food**: Available
   - **Phone**: 98400-00006
   - **Reason**: Closest to Anna University (1.3 km), but slightly higher rent compared to Sri Sai PG.
